# Planet Hunters — Trial 1 Solution

This notebook implements the architecture in `architecture.md`:

1. Drop the 53 shared all-NaN gap cadences.
2. Median-normalise each light curve and remove per-camera/CCD common-mode systematics.
3. Build low-pass (spots/trends) and high-pass (transits/pulsations) views.
4. Extract physics-aware features: statistics, ACF/FFT, folded transit metrics, a fast custom transit search, flare shapes, centroid shifts, quality flags, and stellar metadata.
5. Train a 5-fold stratified LightGBM multiclass model with balanced class weights and early stopping.
6. Tune per-class probability priors on out-of-fold predictions to maximise macro-F1.
7. Average fold models on the test set and write `submission.csv` with exactly 7,000 rows.

The notebook is self-contained. It caches feature chunks under `feature_cache/` so a rerun after a crash is fast.


## Architecture at a glance

- **Preprocessing:** remove the shared 53-cadence gap, median-normalise each star, subtract per-camera/CCD common-mode signals separately for train and test, and build a 49-cadence median-filter high-pass view.
- **Feature families:** robust statistics and half-sector tests; ACF/FFT periodicity and band powers; folded dip metrics; a fast custom box-transit search (depth, SNR, duration, flatness, secondary eclipse, odd-even); flare rise/decay shape; dip-localised centroid shifts; quality-flag/momentum-dump behaviour; stellar metadata and contamination priors.
- **Model:** 5-fold stratified LightGBM multiclass with balanced class weights and early stopping; fold-averaged test probabilities.
- **Decision rule:** coordinate-ascent per-class prior tuning on out-of-fold probabilities to maximise macro-F1, then argmax. This is safer than forcing the priority order; the priority list is still used as a sanity check.
- **Parallelism:** 6 single-threaded worker processes over 250-star chunks, with per-chunk caching under `feature_cache/`.
- **Optional stages:** a priority-aware cascade or a small 1D CNN can be added after this baseline; they are not required for Trial 1 and are intentionally omitted to keep the first submission robust and fast.


## 1. Imports and configuration

Set `QUICK_TEST=True` to run a small smoke test first. Set it to `False` for the final full run.

In [1]:
# Cell 1: imports, paths, and global configuration
import os
# Keep numerical libraries single-threaded inside worker processes.
# LightGBM uses its own n_jobs setting.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.ndimage import median_filter
from scipy.signal import find_peaks, windows
from scipy.stats import skew, kurtosis
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import lightgbm as lgb

warnings.filterwarnings("ignore")

SEED = 42
DT = 0.020833  # 30-minute cadence in days
BANDS = [(0.0417, 0.1), (0.1, 0.5), (0.5, 1.0), (1.0, 3.0), (3.0, 7.0), (7.0, 15.0), (15.0, 50.0)]

# Set True for a quick smoke test; False for the final full submission.
QUICK_TEST = False
TRAIN_LIMIT = 3000 if QUICK_TEST else None
TEST_LIMIT = 1000 if QUICK_TEST else None

N_JOBS = 6          # 6 physical cores; try 12 only if RAM allows
CHUNK_SIZE = 250    # stars per worker job
DATA_DIR = Path("dataset_and_problem_statement")
# Fallback when the notebook is executed from another working directory.
if not DATA_DIR.exists():
    DATA_DIR = Path(r"C:/Users/Kalpana/Documents/GitHub/Planet_hunters/dataset_and_problem_statement")
CACHE_DIR = Path("feature_cache")
CACHE_DIR.mkdir(exist_ok=True)

print("Data directory:", DATA_DIR.resolve())
print("Quick test:", QUICK_TEST)

Data directory: C:\Users\Kalpana\Documents\GitHub\Planet_hunters\dataset_and_problem_statement
Quick test: False


## 2. Load data and validate the shared gap

The 53 all-NaN cadences are identical in train and test. They match `quality == 4`. We remove them once and use the same valid-cadence mask for every star.

In [2]:
# Cell 2: load metadata and memory-map the large arrays
train_flux_path = DATA_DIR / "train_flux.npy"
train_cen_path = DATA_DIR / "train_centroid.npy"
test_flux_path = DATA_DIR / "test_flux.npy"
test_cen_path = DATA_DIR / "test_centroid.npy"
train_stars_path = DATA_DIR / "train_stars.csv"
test_stars_path = DATA_DIR / "test_stars.csv"

train_flux = np.load(train_flux_path, mmap_mode="r")
train_cen = np.load(train_cen_path, mmap_mode="r")
test_flux = np.load(test_flux_path, mmap_mode="r")
test_cen = np.load(test_cen_path, mmap_mode="r")

train_stars = pd.read_csv(train_stars_path)
test_stars = pd.read_csv(test_stars_path)
train_cad = pd.read_csv(DATA_DIR / "train_cadences.csv")
test_cad = pd.read_csv(DATA_DIR / "test_cadences.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train_flux", train_flux.shape, train_flux.dtype)
print("train_centroid", train_cen.shape, train_cen.dtype)
print("test_flux", test_flux.shape, test_flux.dtype)
print("test_centroid", test_cen.shape, test_cen.dtype)
print("train_stars", train_stars.shape, "test_stars", test_stars.shape)

# The gap is identical for every star. Use a small sample to detect it.
train_gap = np.isnan(np.asarray(train_flux[:200])).all(axis=0)
test_gap = np.isnan(np.asarray(test_flux[:200])).all(axis=0)
valid_train = ~train_gap
valid_test = ~test_gap

assert valid_train.sum() == 1262, f"unexpected valid train cadences: {valid_train.sum()}"
assert valid_test.sum() == 1262, f"unexpected valid test cadences: {valid_test.sum()}"
assert np.array_equal(train_gap, test_gap), "train/test gap masks differ"

train_t = train_cad["time_btjd"].values[valid_train].astype(np.float64)
test_t = test_cad["time_btjd"].values[valid_test].astype(np.float64)
train_q = train_cad["quality"].values[valid_train].astype(np.int64)
test_q = test_cad["quality"].values[valid_test].astype(np.int64)

CLASSES = np.array(sorted(train_stars["label"].unique()))
y_all = train_stars["label"].values
n_train = len(train_stars) if TRAIN_LIMIT is None else min(TRAIN_LIMIT, len(train_stars))
n_test = len(test_stars) if TEST_LIMIT is None else min(TEST_LIMIT, len(test_stars))
test_ids = test_stars["id"].values[:n_test]

print("Valid cadences:", valid_train.sum())
print("Classes:", CLASSES)
print("Train rows used:", n_train, "Test rows used:", n_test)
print(train_stars["label"].value_counts().reindex(CLASSES))

train_flux (18000, 1315) float32
train_centroid (18000, 2, 1315) float16
test_flux (7000, 1315) float32
test_centroid (7000, 2, 1315) float16
train_stars (18000, 9) test_stars (7000, 8)
Valid cadences: 1262
Classes: ['background_binary' 'eclipsing_binary' 'flare_star' 'instrument_glitch'
 'planet' 'pulsating_star' 'quiet_star' 'starspots']
Train rows used: 18000 Test rows used: 7000
label
background_binary    1604
eclipsing_binary     2247
flare_star           1804
instrument_glitch    1721
planet               2145
pulsating_star       1973
quiet_star           3624
starspots            2882
Name: count, dtype: int64


## 3. Preprocessing helpers

Each star is divided by its median. Per-camera/CCD common-mode signals are subtracted separately for train and test. A 49-cadence median filter produces the high-pass view used for transits, eclipses, flares, and pulsations.

In [3]:
# Cell 3: normalisation, common-mode removal, and basic statistics
def normalize(flux,cen,stars,n,valid):
    stars=stars.iloc[:n]; f=np.asarray(flux[:n,valid],dtype=np.float32)
    rel=f/np.median(f,axis=1,keepdims=True)-1.0
    key=stars.camera.astype(str)+"_"+stars.ccd.astype(str)
    for _,idx in key.groupby(key).groups.items():
        idx=np.asarray(idx); rel[idx]-=np.median(rel[idx],axis=0)
    hp=rel-median_filter(rel,size=(1,49),mode="nearest")
    c=np.asarray(cen[:n,:,valid],dtype=np.float32); c=c-np.median(c,axis=2,keepdims=True)
    for _,idx in key.groupby(key).groups.items():
        idx=np.asarray(idx); c[idx]-=np.median(c[idx],axis=0)[None,:,:]
    sigma=np.maximum(1.4826*np.median(np.abs(hp-np.median(hp,axis=1,keepdims=True)),axis=1),1e-6)
    def fill(x):
        full=np.full((x.shape[0],len(valid)),np.nan,np.float32); full[:,valid]=x; idx=np.arange(full.shape[1])
        for i in range(full.shape[0]):
            row=full[i]; g=~np.isnan(row); full[i]=np.interp(idx,idx[g],row[g]).astype(np.float32)
        return full
    return rel,hp,c,sigma,fill(rel),fill(hp)

def basic_stats(X,prefix):
    out={}; mean=X.mean(1); med=np.median(X,1)
    out[f'{prefix}_mean']=mean; out[f'{prefix}_std']=X.std(1)
    out[f'{prefix}_mad']=1.4826*np.median(np.abs(X-med[:,None]),1)
    out[f'{prefix}_min']=X.min(1); out[f'{prefix}_max']=X.max(1); out[f'{prefix}_range']=out[f'{prefix}_max']-out[f'{prefix}_min']
    out[f'{prefix}_skew']=skew(X,axis=1); out[f'{prefix}_kurt']=kurtosis(X,axis=1)
    qs=np.percentile(X,[1,5,10,25,50,75,90,95,99],axis=1)
    for j,p in enumerate([1,5,10,25,50,75,90,95,99]): out[f'{prefix}_p{p}']=qs[j]
    for thr in [0.01,0.02,0.05]:
        out[f'{prefix}_fneg{thr}']=(X<-thr).mean(1); out[f'{prefix}_fpos{thr}']=(X>thr).mean(1)
    d=np.diff(X,axis=1); out[f'{prefix}_dstd']=d.std(1)
    out[f'{prefix}_dmad']=1.4826*np.median(np.abs(d-np.median(d,1,keepdims=True)),1)
    xm=X-mean[:,None]; var=(xm*xm).mean(1)
    for lag in [1,5,20]: out[f'{prefix}_acf{lag}']=(xm[:,lag:]*xm[:,:-lag]).mean(1)/(var+1e-12)
    for lag in [1,5,20]: out[f'{prefix}_sf{lag}']=np.mean((X[:,lag:]-X[:,:-lag])**2,1)
    out[f'{prefix}_zcr']=np.sum(np.diff(np.signbit(X),axis=1),1)
    return out

def half_stats(X,prefix):
    h=X.shape[1]//2; out={}
    for j,(a,b) in enumerate([(0,h),(h,None)]):
        Y=X[:,a:b]; out[f'{prefix}_std_h{j}']=Y.std(1)
        out[f'{prefix}_mad_h{j}']=1.4826*np.median(np.abs(Y-np.median(Y,1,keepdims=True)),1)
        out[f'{prefix}_med_h{j}']=np.median(Y,1)
    out[f'{prefix}_std_ratio']=out[f'{prefix}_std_h0']/(out[f'{prefix}_std_h1']+1e-12)
    out[f'{prefix}_med_diff']=out[f'{prefix}_med_h0']-out[f'{prefix}_med_h1']
    return out

## 4. Periodicity and event helpers

These functions compute autocorrelation, FFT band powers, folded dip metrics, connected negative/positive events, and local centroid shifts.

In [4]:
# Cell 4: ACF, FFT, folding, event grouping, and centroid shifts
def acf_feats(X,prefix,max_lag=200):
    N,L=X.shape; x=X-X.mean(1,keepdims=True); nfft=1<<int(np.ceil(np.log2(2*L)))
    F=np.fft.rfft(x,n=nfft,axis=1); ac=np.fft.irfft(F*np.conj(F),n=nfft,axis=1)[:,:L]; ac=ac/(ac[:,0:1]+1e-12)
    lags=np.zeros(N); heights=np.zeros(N)
    for i in range(N):
        row=ac[i,:max_lag]; peaks,_=find_peaks(row[2:],distance=2)
        if len(peaks):
            j=peaks[np.argmax(row[2:][peaks])]+2; lags[i]=j; heights[i]=row[j]
    return {f'{prefix}_acf_lag':lags,f'{prefix}_acf_height':heights},lags*DT

def fft_feats(rel_full,hp_full):
    N,L=rel_full.shape; nfft=4096; win=windows.hann(L,sym=False); freq=np.fft.rfftfreq(nfft,d=DT)
    period=np.where(freq>0,1/np.maximum(freq,1e-12),np.inf); out={}; tops={}
    for name,X in [("low",rel_full),("high",hp_full)]:
        P=np.abs(np.fft.rfft((X-X.mean(1,keepdims=True))*win,n=nfft,axis=1))**2; P[:,0]=0; P=P/(P.sum(1,keepdims=True)+1e-12)
        for lo,hi in BANDS:
            m=(period>=lo)&(period<hi); out[f'fft_{name}_band_{lo}_{hi}']=P[:,m].sum(1)
        p=P+1e-12; out[f'fft_{name}_entropy']=-(p*np.log(p)).sum(1); out[f'fft_{name}_peak']=P.max(1)
        periods=np.full((N,5),np.nan,np.float32); powers=np.zeros((N,5),np.float32)
        for i in range(N):
            row=P[i]; peaks,_=find_peaks(row,distance=3)
            if len(peaks)==0: peaks=np.array([int(np.argmax(row))])
            order=peaks[np.argsort(row[peaks])[::-1][:5]]; periods[i,:len(order)]=1/np.maximum(freq[order],1e-12); powers[i,:len(order)]=row[order]
        tops[name]=(periods,powers)
        for j in range(3): out[f'fft_{name}_p{j+1}']=periods[:,j]; out[f'fft_{name}_pow{j+1}']=powers[:,j]
        out[f'fft_{name}_ratio12']=powers[:,0]/(powers[:,1]+1e-12); out[f'fft_{name}_ratio13']=powers[:,0]/(powers[:,2]+1e-12)
    return out,tops

def folded_metrics(t,y,P,nbins=101):
    if not np.isfinite(P) or P<=0.02: return (0,0,0,0,0,0)
    bins=np.minimum(((t%P)/P*nbins).astype(np.int32),nbins-1)
    cnt=np.bincount(bins,minlength=nbins).astype(np.float32); s=np.bincount(bins,weights=y,minlength=nbins).astype(np.float32)
    prof=np.divide(s,cnt,out=np.full(nbins,np.nan,np.float32),where=cnt>0); med=np.nanmedian(prof); prof=np.where(np.isnan(prof),med,prof)
    imin=int(np.argmin(prof)); depth=float(med-prof[imin])
    if depth<=0: return (0,0,0,0,0,0)
    idx=np.arange(nbins); dist=np.minimum((idx-imin)%nbins,(imin-idx)%nbins); outm=dist>5
    noise=float(np.std(prof[outm])/np.sqrt(max(cnt[outm].mean(),1))); snr=depth/(noise+1e-9); half=med-0.5*depth; width=int(np.sum(prof<half))
    flat=float(np.mean(prof[dist<=max(width//2,1)]<med-0.9*depth)); sec_idx=(imin+nbins//2)%nbins
    sec_depth=float(med-np.median(prof[[(sec_idx+k)%nbins for k in range(-2,3)]]))
    peaks,_=find_peaks(-prof,distance=2); n_dips=int(np.sum((med-prof[peaks])>0.5*depth)) if len(peaks) else 0
    return float(depth),float(snr),width,flat,sec_depth,n_dips

def groups(mask):
    if not mask.any(): return []
    d=np.diff(np.r_[False,mask,False].astype(np.int8)); return list(zip(np.where(d==1)[0],np.where(d==-1)[0]))

def centroid_shift(cx,cy,inm,outm):
    if inm.sum()<1 or outm.sum()<3: return (0,0,0,0,0)
    mx=np.median(cx[inm]); my=np.median(cy[inm]); ox=np.median(cx[outm]); oy=np.median(cy[outm])
    sigx=1.4826*np.median(np.abs(cx[outm]-ox))+1e-6; sigy=1.4826*np.median(np.abs(cy[outm]-oy))+1e-6
    sx=mx-ox; sy=my-oy; return float(sx),float(sy),float(sx/sigx),float(sy/sigy),float(np.hypot(sx/sigx,sy/sigy))

## 5. Fast custom transit search

`astropy` BLS is too slow for 18,000 stars. This implementation folds on a fixed log-spaced period grid plus the strongest FFT peaks, then uses circular box means to measure transit depth, SNR, duration, flatness, secondary eclipse, and odd-even depth differences.

In [5]:
# Cell 5: sinusoid fit and fast box-least-squares-style transit search
def sinusoid_r2(t,y,P):
    if not np.isfinite(P) or P<=0: return 0.0
    w=2*np.pi/P
    A=np.column_stack([np.sin(w*t),np.cos(w*t),np.ones_like(t)])
    coef,*_=np.linalg.lstsq(A,y,rcond=None)
    pred=A@coef
    ss_res=((y-pred)**2).sum(); ss_tot=((y-y.mean())**2).sum()
    return float(1-ss_res/(ss_tot+1e-12))

def transit_features(t,y,sigma,periods,nbins=201,durations=(0.06,0.10,0.15,0.20)):
    base_all=float(np.median(y))
    best=dict(t_depth=0.0,t_snr=0.0,t_power=0.0,t_period=0.0,t_duration=0.0,t_width=0.0,t_flat=0.0,t_sec=0.0,t_oddeven=0.0,t_ntrans=0.0,t_duty=0.0)
    seen=set()
    for P in periods:
        if not np.isfinite(P) or P<0.4 or P>13: continue
        key=round(float(P),4)
        if key in seen: continue
        seen.add(key)
        phase=(t%P)/P
        bins=np.minimum((phase*nbins).astype(np.int32),nbins-1)
        cnt=np.bincount(bins,minlength=nbins).astype(np.float32)
        s=np.bincount(bins,weights=y,minlength=nbins).astype(np.float32)
        prof=np.divide(s,cnt,out=np.full(nbins,base_all,np.float32),where=cnt>0)
        prof[cnt==0]=base_all
        base=float(np.median(prof))
        noise=float(np.std(prof-base)/np.sqrt(max(np.mean(cnt),1.0)))+1e-9
        ntrans=float((t.max()-t.min())/P)
        for dur in durations:
            w=max(1,int(round(dur/P*nbins)))
            if w>=nbins//2: continue
            ext=np.r_[prof,prof[:w]]
            cs=np.cumsum(np.r_[0.0,ext])
            means=(cs[w:w+nbins]-cs[:nbins])/w
            j=int(np.argmin(means))
            depth=float(base-means[j])
            if depth<=0: continue
            snr=depth/noise
            power=snr*np.sqrt(max(ntrans,1.0))
            if power>best['t_power']:
                center=(j+0.5)/nbins
                dphase=np.abs(((phase-center+0.5)%1.0)-0.5)
                inm=dphase<(w/nbins)*1.5
                idx=(np.arange(j,j+w))%nbins
                flat=float(np.mean(prof[idx]<base-0.9*depth)) if w>0 else 0.0
                sec_idx=(np.arange(j+nbins//2,j+nbins//2+w))%nbins
                sec=float(base-np.mean(prof[sec_idx]))
                oddeven=0.0
                if inm.sum()>=6:
                    cyc=np.floor((t-t[0])/P).astype(np.int64)
                    even=inm&(cyc%2==0); odd=inm&(cyc%2==1)
                    if even.sum()>=3 and odd.sum()>=3:
                        de=base-np.median(y[even]); do=base-np.median(y[odd])
                        oddeven=float(abs(de-do)/(depth+1e-12))
                best.update(t_depth=depth,t_snr=float(snr),t_power=float(power),t_period=float(P),t_duration=float(dur),t_width=float(w),t_flat=flat,t_sec=sec,t_oddeven=oddeven,t_ntrans=ntrans,t_duty=float(w/nbins))
    return best

## 6. Per-star feature extraction

This assembles the full 263-feature row for one star: statistics, periodicity, transit candidates, flares, centroid motion, quality flags, and metadata.

In [6]:
# Cell 6: per-star features, metadata priors, and feature-matrix builder
def per_star(rel,hp,c,sigma,t,q,cand_low,cand_high,acf_low_period,period_grid,radius,contamination):
    out={}; cx,cy=c[0],c[1]
    out['low_smooth']=float(np.std(np.diff(rel,2))/(np.std(rel)+1e-12))
    out['high_smooth']=float(np.std(np.diff(hp,2))/(np.std(hp)+1e-12))
    out['low_high_ratio']=float(np.std(rel)/(np.std(hp)+1e-12))
    out['sin_low_r2']=sinusoid_r2(t,rel,cand_low[0] if len(cand_low) else np.nan)
    out['sin_high_r2']=sinusoid_r2(t,hp,cand_high[0] if len(cand_high) else np.nan)
    out['cen_std_x']=cx.std(); out['cen_std_y']=cy.std()
    out['cen_mad_x']=1.4826*np.median(np.abs(cx-np.median(cx))); out['cen_mad_y']=1.4826*np.median(np.abs(cy-np.median(cy)))
    out['cen_range_x']=cx.max()-cx.min(); out['cen_range_y']=cy.max()-cy.min()
    out['cen_corr_xy']=np.corrcoef(cx,cy)[0,1] if cx.std()>0 and cy.std()>0 else 0
    out['cen_corr_fx']=np.corrcoef(rel,cx)[0,1] if rel.std()>0 and cx.std()>0 else 0
    out['cen_corr_fy']=np.corrcoef(rel,cy)[0,1] if rel.std()>0 and cy.std()>0 else 0
    out['cen_abs_corr_f']=max(abs(out['cen_corr_fx']),abs(out['cen_corr_fy']))
    for name,v in [('x',cx),('y',cy)]:
        d=np.diff(v); out[f'cen_maxjump_{name}']=np.max(np.abs(d)) if len(d) else 0
        out[f'cen_njump_{name}']=int(np.sum(np.abs(d)>3*1.4826*np.median(np.abs(d-np.median(d)))+1e-9))
    neg=hp<-3*sigma; gs=groups(neg); depths=[]; durs=[]; areas=[]; flats=[]; syms=[]; shifts=[]; centers=[]; align_dump=0
    for s,e in gs:
        seg=hp[s:e]; dep=float(-seg.min()); depths.append(dep); durs.append(e-s); areas.append(float(-seg.sum())); pk=s+int(np.argmin(seg))
        flats.append(float(np.mean(seg<-0.9*dep)) if dep>0 else 0); syms.append((e-pk)/max(pk-s+1,1))
        inm=np.zeros(len(hp),bool); inm[s:e]=True; shifts.append(centroid_shift(cx,cy,inm,~inm)[4]); centers.append((s+e)//2); align_dump+=int(((q[s:e]&1)!=0).any())
    out['neg_n']=len(gs)
    keys_neg=['neg_maxdepth','neg_meddepth','neg_maxdur','neg_meddur','neg_maxarea','neg_totalarea','neg_maxsnr','neg_flat','neg_medsym','neg_depth_cv','neg_frac_multi','neg_maxshift','neg_medshift','neg_space_med','neg_space_cv','neg_n_dump','neg_single_frac','neg_depth_rel','neg_shift_over_depth']
    if gs:
        out['neg_maxdepth']=max(depths); out['neg_meddepth']=float(np.median(depths)); out['neg_maxdur']=max(durs); out['neg_meddur']=float(np.median(durs))
        out['neg_maxarea']=max(areas); out['neg_totalarea']=float(np.sum(areas)); out['neg_maxsnr']=out['neg_maxdepth']/sigma
        out['neg_flat']=float(np.mean(flats)); out['neg_medsym']=float(np.median(syms)); out['neg_depth_cv']=float(np.std(depths)/(np.mean(depths)+1e-12))
        out['neg_frac_multi']=float(np.mean(np.array(durs)>=2)); out['neg_maxshift']=max(shifts); out['neg_medshift']=float(np.median(shifts))
        out['neg_single_frac']=float(np.mean(np.array(durs)==1)); out['neg_depth_rel']=float(out['neg_maxdepth']/(np.std(rel)+1e-12))
        out['neg_shift_over_depth']=float(out['neg_maxshift']/(out['neg_maxdepth']/sigma+1e-12))
        if len(centers)>1:
            sp=np.diff(centers); out['neg_space_med']=float(np.median(sp)); out['neg_space_cv']=float(np.std(sp)/(np.mean(sp)+1e-12))
        else: out['neg_space_med']=0; out['neg_space_cv']=0
        out['neg_n_dump']=align_dump
    else:
        for k in keys_neg: out[k]=0
    pos=hp>3*sigma; gs=groups(pos); amps=[]; fdurs=[]; decays=[]; rises=[]; syms=[]; areas=[]; shifts=[]; multi=0
    for s,e in gs:
        seg=hp[s:e]; amp=float(seg.max()); pk=s+int(np.argmax(seg)); amps.append(amp); fdurs.append(e-s); rises.append(pk-s+1); decays.append(e-pk); syms.append((e-pk)/max(pk-s+1,1)); areas.append(float(seg.sum()))
        inm=np.zeros(len(hp),bool); inm[s:e]=True; shifts.append(centroid_shift(cx,cy,inm,~inm)[4]); multi+=int(e-s>=2)
    out['flare_n']=len(gs)
    keys_fl=['flare_maxamp','flare_medamp','flare_maxdur','flare_meddur','flare_maxarea','flare_totalarea','flare_maxrise','flare_maxdecay','flare_meddecay','flare_maxasym','flare_frac_multi','flare_maxshift','flare_rate','flare_n5','flare_n8','flare_single','flare_amp_over_sigma']
    if gs:
        out['flare_maxamp']=max(amps); out['flare_medamp']=float(np.median(amps)); out['flare_maxdur']=max(fdurs); out['flare_meddur']=float(np.median(fdurs))
        out['flare_maxarea']=max(areas); out['flare_totalarea']=float(np.sum(areas)); out['flare_maxrise']=max(rises); out['flare_maxdecay']=max(decays); out['flare_meddecay']=float(np.median(decays))
        out['flare_maxasym']=max(syms); out['flare_frac_multi']=multi/len(gs); out['flare_maxshift']=max(shifts); out['flare_n_stable']=int(np.sum(np.array(shifts)<3))
        out['flare_rate']=len(gs)/27.4; out['flare_n5']=int(np.sum(np.array(amps)>5*sigma)); out['flare_n8']=int(np.sum(np.array(amps)>8*sigma)); out['flare_single']=int(np.sum(np.array(fdurs)==1))
        out['flare_amp_over_sigma']=float(out['flare_maxamp']/sigma)
    else:
        for k in keys_fl: out[k]=0
        out['flare_n_stable']=0
    cands=[]
    for P in list(cand_low[:2])+list(cand_high[:2])+[acf_low_period]:
        if np.isfinite(P) and 0.1<=P<=20 and not any(abs(P-Q)/max(P,Q)<0.02 for Q in cands): cands.append(float(P))
    cands=cands[:4]
    for j in range(4):
        P=cands[j] if j<len(cands) else np.nan; dm=folded_metrics(t,hp,P) if np.isfinite(P) else (0,0,0,0,0,0)
        out[f'fold{j}_period']=P if np.isfinite(P) else 0
        for k,val in zip(['depth','snr','width','flat','sec','ndips'],dm): out[f'fold{j}_{k}']=val
    P=cand_high[0] if len(cand_high) else np.nan
    if np.isfinite(P) and P>0.02:
        bins=np.minimum(((t%P)/P*101).astype(np.int32),100); cnt=np.bincount(bins,minlength=101); ssum=np.bincount(bins,weights=hp,minlength=101); prof=ssum/(cnt+1e-9); imin=int(np.argmin(prof))
        inm=(bins==imin)|(bins==(imin-1)%101)|(bins==(imin+1)%101); sx,sy,sxr,syr,mag=centroid_shift(cx,cy,inm,~inm)
        out['fold_cen_shift']=mag; out['fold_cen_sx']=sxr; out['fold_cen_sy']=syr
    else: out['fold_cen_shift']=0; out['fold_cen_sx']=0; out['fold_cen_sy']=0
    dump=(q&1)!=0; scat=(q&2)!=0
    out['qi_frac_dump']=dump.mean(); out['qi_frac_scat']=scat.mean(); out['qi_n_nonzero']=int((q!=0).sum())
    out['qi_dump_offset']=float(np.median(rel[dump])-np.median(rel[~dump])) if dump.sum() else 0
    out['qi_scat_offset']=float(np.median(rel[scat])-np.median(rel[~scat])) if scat.sum() else 0
    d=np.diff(rel); out['qi_jump_max']=float(np.max(np.abs(d))); out['qi_njump5']=int(np.sum(np.abs(d)>5*sigma)); out['qi_njump10']=int(np.sum(np.abs(d)>10*sigma))
    out['qi_jump_ratio']=float(out['qi_jump_max']/sigma)
    out['fold_shift_over_depth']=float(out.get('fold_cen_shift',0)/(out.get('fold0_depth',0)/sigma+1e-12))
    out['cen_shift_contam']=float(out.get('neg_maxshift',0)*contamination)
    out['neg_depth_contam']=float(out.get('neg_maxdepth',0)*contamination)
    periods=np.r_[period_grid, cand_low[:2], cand_high[:5], acf_low_period]
    tf=transit_features(t,hp,sigma,periods)
    out.update(tf)
    out['t_depth_over_sigma']=float(out['t_depth']/sigma)
    out['t_sec_over_depth']=float(out['t_sec']/(out['t_depth']+1e-12))
    out['t_depth_over_expected']=float(out['t_depth']/((0.10045/max(radius,0.01))**2+1e-12))
    out['t_depth_contam']=float(out['t_depth']*contamination)
    out['t_shift_ratio']=float(out.get('fold_cen_shift',0)/(out['t_depth']/sigma+1e-12))
    return out

def metadata_frame(stars):
    m=pd.DataFrame(index=stars.index)
    m['tmag']=stars.tmag.astype(float); m['teff']=stars.teff.astype(float); m['logg']=stars.logg.astype(float); m['radius']=stars.radius.astype(float); m['contamination']=stars.contamination.astype(float)
    m['log_radius']=np.log10(stars.radius.clip(lower=0.01)); m['log_teff']=np.log10(stars.teff.clip(lower=100))
    m['lum']=(stars.radius**2)*(stars.teff/5772)**4; m['density']=stars.logg-np.log10(stars.radius.clip(lower=0.01))
    m['is_m']=(stars.teff<4000).astype(float); m['is_giant']=(stars.logg<3.5).astype(float); m['contam_radius']=stars.contamination*stars.radius
    key=stars.camera.astype(str)+"_"+stars.ccd.astype(str); d=pd.get_dummies(key,prefix='ccd',dtype=np.float32)
    return pd.concat([m,d],axis=1)

def build_features(flux,cen,stars,t,q,valid,n):
    t0=time.time(); rel,hp,c,sigma,rel_full,hp_full=normalize(flux,cen,stars,n,valid)
    S={}; S.update(basic_stats(rel,"low")); S.update(basic_stats(hp,"high")); S.update(half_stats(rel,"low")); S.update(half_stats(hp,"high"))
    a,ap=acf_feats(rel,"low"); S.update(a); b,bp=acf_feats(hp,"high"); S.update(b); f,tops=fft_feats(rel_full,hp_full); S.update(f)
    P_low,Pow_low=tops['low']; P_high,Pow_high=tops['high']
    period_grid=np.exp(np.linspace(np.log(0.3),np.log(15.0),90))
    radius=stars.iloc[:n].radius.values.astype(float)
    contamination=stars.iloc[:n].contamination.values.astype(float)
    rec=[per_star(rel[i],hp[i],c[i],sigma[i],t,q,P_low[i],P_high[i],ap[i],period_grid,radius[i],contamination[i]) for i in range(n)]
    X=pd.concat([pd.DataFrame(S),pd.DataFrame.from_records(rec),metadata_frame(stars.iloc[:n])],axis=1)
    X=X.replace([np.inf,-np.inf],np.nan)
    print("features",X.shape,"sec",time.time()-t0)
    return X

## 7. Parallel feature extraction with caching

Windows uses process-based parallelism. Each worker loads only its chunk through mmap, so peak RAM stays low. Completed chunks are cached to `feature_cache/` and reused after a crash.

In [7]:
# Cell 7: chunk worker and parallel driver
def _extract_chunk(flux_path, cen_path, stars_path, start, end, t, q, valid, split, cache_dir):
    """Extract features for one chunk and cache the result."""
    cache_file = Path(cache_dir) / f"{split}_{start:05d}_{end:05d}.pkl"
    if cache_file.exists():
        return pd.read_pickle(cache_file)

    # mmap slices avoid copying the full 90 MB array into every worker.
    flux = np.load(flux_path, mmap_mode="r")[start:end]
    cen = np.load(cen_path, mmap_mode="r")[start:end]
    stars = pd.read_csv(stars_path).iloc[start:end].reset_index(drop=True)

    X = build_features(flux, cen, stars, t, q, valid, end - start)
    X.index = np.arange(start, end)
    X.to_pickle(cache_file)
    return X


def extract_features(flux_path, cen_path, stars_path, t, q, valid, n,
                     split, chunk=CHUNK_SIZE, n_jobs=N_JOBS, cache_dir=CACHE_DIR):
    """Run feature extraction over chunks in parallel and concatenate in order."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    ranges = [(s, min(s + chunk, n)) for s in range(0, n, chunk)]
    print(f"{split}: {len(ranges)} chunks of up to {chunk} stars, n_jobs={n_jobs}")

    parts = Parallel(n_jobs=n_jobs, backend="loky", verbose=5)(
        delayed(_extract_chunk)(
            str(flux_path), str(cen_path), str(stars_path),
            s, e, t, q, valid, split, str(cache_dir)
        )
        for s, e in ranges
    )
    X = pd.concat(parts, axis=0).sort_index()
    print(f"{split}: feature matrix {X.shape}")
    return X

## 8. Build the train and test feature matrices

The first full run takes roughly 4–8 minutes for train and 2–3 minutes for test on a 6-core CPU. Later runs load cached chunks and are much faster.

In [8]:
# Cell 8: build train and test features
t0 = time.time()
X_train = extract_features(
    train_flux_path, train_cen_path, train_stars_path,
    train_t, train_q, valid_train, n_train, "train"
)
y_train = pd.Series(y_all[:n_train], name="label")

X_test = extract_features(
    test_flux_path, test_cen_path, test_stars_path,
    test_t, test_q, valid_test, n_test, "test"
)

# Align columns in case a rare feature is constant in one split.
X_test = X_test.reindex(columns=X_train.columns, fill_value=0.0)

# Save combined matrices for fast reuse.
X_train.to_pickle(CACHE_DIR / "X_train.pkl")
X_test.to_pickle(CACHE_DIR / "X_test.pkl")
print("X_train", X_train.shape, "X_test", X_test.shape, "elapsed", round(time.time() - t0, 1), "s")

train: 72 chunks of up to 250 stars, n_jobs=6


[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.


[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:   25.9s


[Parallel(n_jobs=6)]: Done  60 tasks      | elapsed:  3.6min


[Parallel(n_jobs=6)]: Done  72 out of  72 | elapsed:  4.3min finished
[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.


train: feature matrix (18000, 263)
test: 28 chunks of up to 250 stars, n_jobs=6


[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:   21.1s


[Parallel(n_jobs=6)]: Done  23 out of  28 | elapsed:  1.4min remaining:   18.3s


test: feature matrix (7000, 263)
X_train (18000, 263) X_test (7000, 263) elapsed 364.4 s


[Parallel(n_jobs=6)]: Done  28 out of  28 | elapsed:  1.7min finished


## 9. LightGBM cross-validation

5-fold stratified CV gives out-of-fold probabilities for honest threshold tuning. Each fold model also predicts the test set; the final test probability is the fold average.

In [9]:
# Cell 9: 5-fold LightGBM with early stopping
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
classes = np.array(sorted(y_train.unique()))
oof = np.zeros((len(X_train), len(classes)), dtype=np.float32)
test_proba = np.zeros((len(X_test), len(classes)), dtype=np.float32)
best_iters = []
feature_importance = np.zeros(X_train.shape[1], dtype=np.float64)

LGB_PARAMS = dict(
    objective="multiclass",
    num_class=len(classes),
    metric="multi_logloss",
    learning_rate=0.04,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.75,
    reg_lambda=5.0,
    n_estimators=2000,
    n_jobs=N_JOBS,
    verbosity=-1,
    random_state=SEED,
    class_weight="balanced",
)

for fold, (tr, va) in enumerate(skf.split(X_train, y_train), 1):
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(
        X_train.iloc[tr], y_train.iloc[tr],
        eval_set=[(X_train.iloc[va], y_train.iloc[va])],
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    assert np.array_equal(model.classes_, classes)
    oof[va] = model.predict_proba(X_train.iloc[va])
    test_proba += model.predict_proba(X_test) / skf.n_splits
    best_iters.append(model.best_iteration_)
    feature_importance += model.feature_importances_
    print(f"fold {fold}: best_iter={model.best_iteration_}")

baseline_pred = classes[oof.argmax(axis=1)]
baseline_f1 = f1_score(y_train, baseline_pred, average="macro")
print("OOF macro-F1 (argmax):", round(baseline_f1, 4))
print(classification_report(y_train, baseline_pred, digits=3))
print("confusion matrix:\n", confusion_matrix(y_train, baseline_pred, labels=classes))

imp = pd.Series(feature_importance, index=X_train.columns).sort_values(ascending=False)
print("Top 30 features:\n", imp.head(30).to_string())
np.save(CACHE_DIR / "oof.npy", oof)
np.save(CACHE_DIR / "test_proba.npy", test_proba)

fold 1: best_iter=192


fold 2: best_iter=214


fold 3: best_iter=177


fold 4: best_iter=226


fold 5: best_iter=234
OOF macro-F1 (argmax): 0.6537


                   precision    recall  f1-score   support

background_binary      0.683     0.545     0.606      1604
 eclipsing_binary      0.860     0.665     0.750      2247
       flare_star      0.768     0.625     0.689      1804
instrument_glitch      0.827     0.823     0.825      1721
           planet      0.517     0.414     0.460      2145
   pulsating_star      0.816     0.715     0.762      1973
       quiet_star      0.502     0.781     0.611      3624
        starspots      0.541     0.511     0.525      2882

         accuracy                          0.640     18000
        macro avg      0.689     0.635     0.654     18000
     weighted avg      0.663     0.640     0.641     18000

confusion matrix:
 [[ 874  170   21   16  154   38  221  110]
 [ 225 1494   22   15  163   22  174  132]
 [  11    8 1128   19   48   25  315  250]
 [   9    3   14 1416   51   33  123   72]
 [ 103   42   34   42  889   34  642  359]
 [  17    4   12   34   38 1410  390   68]
 [  19    6 

## 10. Tune per-class priors for macro-F1

Macro-F1 weights every class equally. A small coordinate ascent over per-class probability multipliers often improves rare/hard classes without retraining.

In [10]:
# Cell 10: coordinate-ascent prior tuning on OOF probabilities
def tune_class_priors(proba, y, classes, n_passes=3):
    weights = np.ones(len(classes), dtype=np.float64)
    baseline = f1_score(y, classes[proba.argmax(axis=1)], average="macro")
    best = baseline
    grid = [0.35, 0.5, 0.65, 0.8, 1.0, 1.2, 1.45, 1.75, 2.1, 2.6, 3.2]
    for _ in range(n_passes):
        improved = False
        for j in range(len(classes)):
            for multiplier in grid:
                trial = weights.copy()
                trial[j] = multiplier
                pred = classes[(proba * trial).argmax(axis=1)]
                score = f1_score(y, pred, average="macro")
                if score > best + 1e-6:
                    best = score
                    weights = trial
                    improved = True
        if not improved:
            break
    return weights, baseline, best

prior_weights, prior_baseline, prior_tuned = tune_class_priors(oof, y_train, classes)
print("OOF macro-F1 before tuning:", round(prior_baseline, 4))
print("OOF macro-F1 after tuning: ", round(prior_tuned, 4))
print("Class order:", list(classes))
print("Prior weights:", np.round(prior_weights, 3))

tuned_pred = classes[(oof * prior_weights).argmax(axis=1)]
print(classification_report(y_train, tuned_pred, digits=3))
print("confusion matrix:\n", confusion_matrix(y_train, tuned_pred, labels=classes))

OOF macro-F1 before tuning: 0.6537
OOF macro-F1 after tuning:  0.6589
Class order: [np.str_('background_binary'), np.str_('eclipsing_binary'), np.str_('flare_star'), np.str_('instrument_glitch'), np.str_('planet'), np.str_('pulsating_star'), np.str_('quiet_star'), np.str_('starspots')]
Prior weights: [1.   0.8  0.65 0.5  0.8  0.65 1.2  1.  ]


                   precision    recall  f1-score   support

background_binary      0.664     0.569     0.613      1604
 eclipsing_binary      0.875     0.653     0.748      2247
       flare_star      0.850     0.594     0.699      1804
instrument_glitch      0.899     0.779     0.835      1721
           planet      0.577     0.372     0.453      2145
   pulsating_star      0.883     0.682     0.769      1973
       quiet_star      0.483     0.865     0.620      3624
        starspots      0.543     0.528     0.535      2882

         accuracy                          0.644     18000
        macro avg      0.722     0.630     0.659     18000
     weighted avg      0.689     0.644     0.647     18000

confusion matrix:
 [[ 912  149   13    8  123   19  265  115]
 [ 255 1467   13    5  146   13  209  139]
 [  11    8 1071    9   29   17  378  281]
 [  16    3    8 1341   52   24  188   89]
 [ 119   35   16   24  799   16  770  366]
 [  16    5    6   19   25 1345  483   74]
 [  18    3 

## 11. Write the submission

The submission uses the fold-averaged test probabilities multiplied by the tuned priors. IDs are kept in the exact order of `test_stars.csv`, which is also the order of `sample_submission.csv`.

In [11]:
# Cell 11: predict, validate, and write submission.csv
test_prob_tuned = test_proba * prior_weights
test_pred = classes[test_prob_tuned.argmax(axis=1)]

submission = pd.DataFrame({"id": test_ids, "label": test_pred})
out_path = Path("submission_quick.csv") if QUICK_TEST else Path("submission.csv")
submission.to_csv(out_path, index=False)

print("Wrote", out_path.resolve())
print("Rows:", len(submission))
print(submission["label"].value_counts())

if not QUICK_TEST:
    assert len(submission) == 7000, "submission must have exactly 7000 rows"
    assert submission.columns.tolist() == ["id", "label"]
    assert (submission["id"].values == sample_sub["id"].values).all(), "ID order mismatch"
    assert set(submission["label"]).issubset(set(CLASSES)), "unknown label found"
    print("submission.csv passed all checks")

Wrote C:\Users\Kalpana\Documents\GitHub\Planet_hunters\submission.csv
Rows: 7000
label
quiet_star           1824
starspots            1413
instrument_glitch     767
pulsating_star        634
eclipsing_binary      632
planet                616
background_binary     610
flare_star            504
Name: count, dtype: int64
submission.csv passed all checks


## Run instructions

### Environment

From PowerShell in `C:\Users\Kalpana\Documents\GitHub\Planet_hunters`:

```powershell
.\myenv\Scripts\Activate.ps1
python -m pip install --upgrade lightgbm
```

`lightgbm` and `astropy` are already installed in the current environment. The notebook does **not** use astropy BLS.

### Run

1. Open `planet_hunters_solution.ipynb` in VS Code or Jupyter.
2. Run the cells from top to bottom.
3. For a quick smoke test, set `QUICK_TEST = True` in Cell 1. This writes `submission_quick.csv` with 1,000 rows.
4. For the final entry, set `QUICK_TEST = False` and run all cells. It writes `submission.csv` with 7,000 rows.

Command-line alternative:

```powershell
jupyter nbconvert --to notebook --execute planet_hunters_solution.ipynb --output planet_hunters_solution_executed.ipynb --ExecutePreprocessor.timeout=3600
```

### Expected resources

- Full train feature extraction: ~4–8 minutes on 6 cores.
- Full test feature extraction: ~2–3 minutes.
- 5-fold LightGBM: ~5–10 minutes.
- Peak RAM: roughly 2–4 GB with chunking; close other applications if you have 8 GB.
- Total: about 10–20 minutes on the first run; cached reruns are much faster.

### Submission strategy

- Trust OOF macro-F1 and robustness over the public LB; public LB is only 30% of the test set.
- Use at most 3 of the 5 daily uploads: one baseline, one after prior tuning, and one only if a meaningful CV improvement is confirmed.
- Keep `architecture.md`, this notebook, and the generated `feature_cache/` for reproducibility.

### Common fixes

- If a worker crashes, reduce `CHUNK_SIZE` to 100 and `N_JOBS` to 4.
- If LightGBM is missing, `python -m pip install lightgbm`; a slower fallback is `sklearn.ensemble.HistGradientBoostingClassifier`.
- If RAM is tight, run train and test feature extraction in separate sessions; the cache makes this safe.
- If a class is missing from a fold, use `StratifiedKFold` (already used) or merge extremely rare classes.